In [ ]:
# THINK OF SOMETHING TO BE CREATIVE
# MAKE THE CODE SCALABLE 
# OR SUPER ACCURATE

In [6]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, List, Dict
from langchain_groq import ChatGroq
import json

In [ ]:
class AgentState(TypedDict):
    user_profile_path: str
    product_catalog_path: str
    unique_categories: List[str]
    product_catalogue: dict
    user_profile: dict
    justification: str
    recommended_category: str
    top3_products: List[str]
    error: str 

In [17]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or mixtral-8x7b
    temperature=0.6,
    api_key = "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"
)

# give a prompt to take the user history and the unique categories and choose a likeable category
category_prompt = PromptTemplate.from_template("""
You are an intelligent shopping recommendation assistant.

You will receive:
1. The user's purchase history as a JSON list.
2. A list of unique product categories available in the catalog.

Your task:
- Analyze the user's purchase history.
- Identify categories the user has **not purchased from** before.
- If all categories have been purchased from, choose one the user has **purchased from the least**.
- Choose **one category** that the user might be most interested in exploring next.
- Generate a **catchy, persuasive justification** in natural language explaining *why* this category fits the user’s preferences or lifestyle.

Return your final answer **strictly** in the following JSON format:

{{
  "recommended_category": "<category_name>",
  "justification": "<short persuasive message>"
}}

### USER HISTORY:
{user_history}

### AVAILABLE CATEGORIES:
{available_categories}

Now respond only with the JSON object described above — no extra text.
""")
category_chain = LLMChain(llm=llm, prompt=category_prompt)



In [ ]:
#node 1
# add conditional edges
def category_extraction(state ,user_profile_path ,product_catalog_path ):
    state["user_profile_path"] = user_profile_path
    state["product_catalog_path"] = product_catalog_path
    # read user profile
    try:
        with open(state["user_profile_path"], "r") as f:
            state["user_profile"] = json.load(f)
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['user_profile_path']} not found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['user_profile_path']}."
        return state

    # read product catalogue
    try:
        with open(state["product_catalog_path"], "r") as f:
            state["product_catalogue"] = json.load(f)
            state["unique_categories"] = sorted({item["category"] for item in state["product_catalogue"]})
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['product_catalog_path']} found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['product_catalog_path']}."
        return state
    return state


In [ ]:
# if no error go to node 2
# node 2 
from pydantic import BaseModel, ValidationError

class CategoryRecommendation(BaseModel):
    recommended_category: str = None
    justification: str = None

def category_recommendation(state):
    try:
        # Run the LLM chain with provided user data
        output = chain.run(
            user_history=state["user_profile"]["history"],
            available_categories=state["unique_categories"]
        )

        # Parse JSON
        data = json.loads(output.strip())

        # Validate using your Pydantic model
        validated = CategoryRecommendation(**data)

        # Additional safeguard: ensure both fields are not empty or whitespace
        if not validated.recommended_category.strip() or not validated.justification.strip():
            state["error"] = "❌ One or more fields are empty or invalid in LLM output."
            return state

        # Assign validated values to state
        state["recommended_category"] = validated.recommended_category.strip()
        state["justification"] = validated.justification.strip()


        # CHECK FOR CORRECTNESS

    except json.JSONDecodeError:
        state["error"] = "❌ Invalid JSON format in LLM output."
        return state
    except ValidationError as e:
        state["error"] = f"❌ Invalid LLM output: {str(e)}"
        return state
    except Exception as e:
        state["error"] = f"❌ Unexpected error: {str(e)}"
        return state

    # If everything passes, clear error and return
    return state


In [ ]:
from langchain.prompts import PromptTemplate

recommendation_prompt = PromptTemplate.from_template("""
You are a smart shopping assistant.

You are given:
- The user's purchase history: {user_history}
- A list of available products: {products}
- justification message to convince the user to explore the category : {justification}

Steps to follow:
1. Calculate the user's average spending based on their purchase history.
2. Rank the products based on how close their price is to this average and any inferred preferences of the user.
3. Choose the **top 3 closest products** to recommend.
4. Write a short, friendly **message** summarizing your recommendations for the user and mix it with the justification message.
5. Also, return a **JSON object** containing the selected products.

Your response **must follow this exact format**:

Message:
<Your friendly, natural language message to the user>

JSON:
{{
  "average_price": "<computed_average_as_number>",
  "top_3_products": [
    {{"product_id": "<id1>", "product_name": "<name1>", "price": <price1>}},
    {{"product_id": "<id2>", "product_name": "<name2>", "price": <price2>}},
    {{"product_id": "<id3>", "product_name": "<name3>", "price": <price3>}}
  ]
}}

Now respond strictly following this format.
""")


In [ ]:
from langchain.tools import tool
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

@tool("average_calculator", return_direct=True)
def average_calculator(numbers: str) -> str:
    """
    Calculates the average of a comma-separated list of numbers.
    Example input: "120, 80, 200"
    """
    try:
        nums = [float(n.strip()) for n in numbers.split(",") if n.strip()]
        if not nums:
            return "No valid numbers provided."
        avg = sum(nums) / len(nums)
        return f"The average is {avg:.2f}"
    except Exception as e:
        return f"Error calculating average: {str(e)}"

tools = [
    Tool(
        name="Average Calculator",
        func=average_calculator,
        description="Use this tool to calculate the average of the prices of the user's history purchased products"
    )


memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)



agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

In [ ]:
def product_recommender(state):
    available_products = [item for item in state["product_catalogue"] if state["recommended_category"] == item["category"]]
    print(available_products)
    
    input_prompt = recommendation_prompt.format(
    user_history=state["user_profile"]["history"],
    products=available_products,
    justification = state["justification"]
    )
    
    
    

In [9]:
graph.add_conditional_edges(
    "recommend_category_node",
    lambda state: "error_handler" if "error" in state else "recommend_products_node"
)

✅ Loaded user profile for user_id: A123
{'user_id': 'A123', 'history': [{'product': 'Wireless Headphones X200', 'category': 'electronics', 'price': 199.99}, {'product': 'Smartwatch Fit S3', 'category': 'electronics', 'price': 249.99}, {'product': 'Running Shoes SprintX', 'category': 'sportswear', 'price': 89.99}, {'product': 'Sports Backpack 25L', 'category': 'sportswear', 'price': 49.99}, {'product': 'Fitness Tracker Watch', 'category': 'sportswear', 'price': 99.99}, {'product': 'Mystery Novel: Hidden Truths', 'category': 'books', 'price': 14.99}, {'product': 'Science Fiction: Galactic Wars', 'category': 'books', 'price': 19.99}, {'product': 'Cookbook: World Cuisines', 'category': 'books', 'price': 29.99}, {'product': 'Thriller: The Last Escape', 'category': 'books', 'price': 22.99}, {'product': 'Data Science Handbook', 'category': 'books', 'price': 59.99}]}


In [7]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplatep

# Create the LLM instance
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or mixtral-8x7b
    temperature=0.6,
    api_key = "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"
)

prompt = PromptTemplate.from_template("Recommend a product for someone who likes {category}")
chain = LLMChain(llm=llm, prompt=prompt)

print(chain.run(category="electronics"))

For someone who likes electronics, I'd recommend the following product:

**Raspberry Pi 4 Model B**

The Raspberry Pi 4 Model B is a compact, affordable, and highly versatile single-board computer that's perfect for electronics enthusiasts. It's a great tool for learning programming, experimenting with DIY projects, and building innovative devices.

Here are some features that make it an excellent choice:

1. **Quad-core processor**: The Raspberry Pi 4 Model B comes with a powerful quad-core Cortex-A72 CPU, which provides a significant boost in performance compared to its predecessors.
2. **4GB or 8GB RAM**: You can choose from two RAM options, depending on your needs. The 4GB model is perfect for general use, while the 8GB model is ideal for more demanding applications.
3. **Dual-display support**: The Raspberry Pi 4 Model B can support two 4K displays at 30Hz, making it perfect for multimedia projects, gaming, and more.
4. **Gigabit Ethernet**: The board features a built-in gigabit E

In [ ]:
groq_api = "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"

In [5]:
from groq import Groq

client = Groq(api_key="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")

models = client.models.list()

for model in models.data:
    print(model.id)

playai-tts-arabic
playai-tts
qwen/qwen3-32b
meta-llama/llama-4-maverick-17b-128e-instruct
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
openai/gpt-oss-20b
whisper-large-v3-turbo
moonshotai/kimi-k2-instruct
whisper-large-v3
groq/compound-mini
llama-3.3-70b-versatile
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
groq/compound
meta-llama/llama-guard-4-12b


In [ ]:
# Create the graph
graph = StateGraph(ShoppingState)

# Add nodes
graph.add_node("category_agent", category_agent)
graph.add_node("product_agent", product_agent)

# Define edges (category -> product -> END)
graph.add_edge("category_agent", "product_agent")
graph.add_edge("product_agent", END)

# Set the entry point
graph.set_entry_point("category_agent")

# Compile the graph
shopping_graph = graph.compile()

In [ ]:
# el haye7sal kal aaty 
# e7na hana5od el user history we nedeeha lel agent
# then el agent hayshoof random category min el categories eli mosh mawgooden fel buyers history